#### Script for clustering using k-medoid

In order to use this script, a hdf5 result file with computed energy consumption per monte carlo load case is needed as input

In [ ]:
%load_ext autoreload
%autoreload 2

import os
os.chdir("../../../")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("FST_bw.mplstyle")
from src.postprocessing.utils_plots import create_standard_list

## Plot

In [ ]:
def get_filenames_optimisation_type(folder_path, cs_names, optimisation_type="Topology"):
    paths = [folder_path + cs + "/" for cs in cs_names]

    standard_dict_list = create_standard_list(paths, optimisation_type, False)
    
    files = []
    for cs, x in zip(cs_names, standard_dict_list):
        if len(x) == 1:
            files.append()
        else:
            files.extend([folder_path + cs + "/" + str(y) for y in x["filename"]])
    
    return files

In [ ]:
folder_path1 = "results/off/combined/max_velocity_5_10ms_max_height_04m/Monte_Carlo_Validation_acoustics/"

cs_names_all = ["VAV-VPC"] #,  "DF-CPC"]

markerstyle = ["o","D", "s","v","^","P"]

ticknames_all = ["KONST/\nZENTRAL", "VAR/\nZENTRAL", "KONST/\nVERTEILT"]
ticknames_min_noise = [ticknames_all[1], ticknames_all[-1]]

ticknames_all = ["VAR/\nZENTRAL"]*2

files1 = get_filenames_optimisation_type(folder_path1, cs_names_all, "Configuration")

In [ ]:
files1

In [ ]:
import h5py
import pandas as pd

sampled_power_consumption = f"/Postprocessing/Multi Load Case Comparison/Points {128}"   # change to your dataset path

fanresults = "/Postprocessing/FanResults"

# exclude_cols = ['time share', 'power consumption in W', 'power consumption lower bound in W']

exclude_cols = ['sound pressure gap in dB', 'time share', 'power consumption in W', 'power consumption lower bound in W']



def load_optimisation_results(path):
    print(path)
    with h5py.File(path, "r") as f:

        scenario = f["Optimisation Components/Set/Scenarios"][:].astype(int)

        volume_flow = {int(s): float() for s in scenario}
        for s in scenario:
            max_volume_flow = np.max(np.array(f[f"Optimisation Components/Parameter/Scenario/{s}/volume_flow"]["value"]))
            volume_flow[s] = max_volume_flow

        # compute optimisation problems power consumption and get time share for comparison

        fan_power = pd.DataFrame(f[fanresults][...])
        fan_power["Scenario"] = fan_power["Scenario"].astype(int)
        fan_power = fan_power.groupby('Scenario')[['PowerConsumption', 'VolumeFlow']].sum().reset_index()
        fan_power = fan_power.set_index("Scenario")

        fan_power.index.name = "Scenarios"


        time_share = f["Optimisation Components/Parameter/time_share"][...]
        time_share = pd.DataFrame(time_share, columns=['Scenarios', 'value'])
        time_share['Scenarios'] = time_share['Scenarios'].astype(int) # time_share['Scenarios'] = time_share['Scenarios'].astype('U').str.strip().replace('', np.nan).astype(float).astype(int)
        time_share['value'] = time_share['value'].astype(float)
        time_share = time_share[['Scenarios', 'value']]
        time_shares = time_share.set_index("Scenarios")["value"]

        power_consumption = fan_power.join(time_shares.rename('TimeShare'))

        power_consumption["VolumeFlow"] = power_consumption.index.map(volume_flow)

    return power_consumption


def load_postprocessing_results(path):
    print(path)
    with h5py.File(path, "r") as f:
        dset = f[sampled_power_consumption]
        arr = dset[...]   # or dset[:]
        df = pd.DataFrame.from_records(arr)
       
        df["total volume flow in m³/h"] = df.drop(columns=exclude_cols).sum(axis=1)
    return df

In [ ]:
dfs,dfs2 = [],[]


for path in files1:
    df = load_postprocessing_results(path)
    dfs.append(df)

    df2 = load_optimisation_results(path)
    dfs2.append(df2)

    # opt_fan_powers.append(opt_fan_power)

In [ ]:
from sklearn_extra.cluster import KMedoids
from sklearn.preprocessing import StandardScaler
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots()

n_clusters = 7

i = 0

x = dfs[i]["total volume flow in m³/h"].to_numpy()
y = dfs[i]["power consumption in W"].to_numpy()

# NaNs entfernen
mask = np.isfinite(x) & np.isfinite(y)
x = x[mask]
y = y[mask]

X = np.column_stack([x, y])

# Skalieren, weil Q und P unterschiedliche Einheiten/Skalen haben
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# K-Medoids
kmedoids = KMedoids(
    n_clusters=n_clusters,
    metric="euclidean",
    init="k-medoids++",
    method="pam",
    random_state=0
)

labels = kmedoids.fit_predict(X_scaled)

# Medoids sind echte Punkte aus X
medoid_indices = kmedoids.medoid_indices_
medoids = X[medoid_indices]

# Cluster plotten
ax.scatter(
    x,
    y,
    c=labels,
    cmap="tab10",
    s=10,
    edgecolor="black",
    linewidth=0.3,
    marker=markerstyle_curr[i],
    label=cs_names_all[i],
    zorder=i,
)

# Medoids markieren
ax.scatter(
    medoids[:, 0],
    medoids[:, 1],
    marker="x",
    s=80,
    color="black",
    linewidth=1.5,
    label="Medoids",
    zorder=i + 10,
)

ax.set_xlabel("total volume flow in m³/h")
ax.set_ylabel("power consumption in W")
ax.legend()



labels = kmedoids.fit_predict(X_scaled)

medoid_indices = kmedoids.medoid_indices_      # positions in df_valid / X
medoid_row_indices = dfs[i].index[medoid_indices]  # original dfs[i] indices
medoids = X[medoid_indices]

medoid_df = dfs[i].loc[medoid_row_indices].copy()
medoid_df["cluster"] = np.arange(n_clusters)
medoid_df["cluster_weight"] = np.bincount(labels, minlength=n_clusters) / len(labels)

print(medoid_df[
    [
        "cluster",
        "cluster_weight",
        "total volume flow in m³/h",
        "power consumption in W",
    ]
])